# 04 — Process NSS Data for Exeter

Take the NSS files downloaded from the Office for Students (OfS), keep only the rows we care about, and turn 26 question-level rows per subject into 8 theme-level scores.

**Inputs:**  `data/raw/nss/NSS{23,24,25}_10007792_{r,t}.csv`  — three years × two cuts.
  - `_r` = registered at Exeter  (the cut we use for the analysis)
  - `_t` = taught at Exeter      (used as a robustness check in notebook 06)

**Outputs (to `data/processed2/`):**
- `nss_exeter_long.csv`           — registered cut, one row per (year, subject, theme)
- `nss_exeter_wide.csv`           — registered cut, one row per (year, subject), 8 theme columns
- `nss_exeter_taughtat_long.csv`  — taught-at cut, long form
- `nss_exeter_taughtat_wide.csv`  — taught-at cut, wide form  *(new in v2)*
- `dept_to_nss_subject.csv`       — module dept code → NSS subject mapping (string fixes from v1)

## Fixes from v1

1. **Strip whitespace from Subject names.** v1's NSS data had `'English studies'` and `'English studies '` (trailing space) as separate subjects. We strip and group them.
2. **Fix the dept→NSS-subject mapping**:
   - `GEO` was mapped to `'Geographical and environmental studies'` — the real NSS string is `'Geography, earth and environmental studies'`.
   - `EFP` (Education) and `LIB` (Liberal arts) were mapped to NSS subjects that **don't exist** in the OfS data, so those rows silently fell out. We remove them with a comment.
3. **Save the taught-at cut in wide form too**, so notebook 06 can do a 1-line robustness check.


## 1. Setup

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / 'data' / 'raw').exists() else HERE.parent

NSS_RAW = PROJECT_ROOT / 'data' / 'raw' / 'nss'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

nss_files = sorted(NSS_RAW.glob('*.csv'))
print('NSS files found:')
for f in nss_files:
    print(f'  - {f.name}')

## 2. Map NSS questions to themes

The NSS publishes 26 numbered questions. The OfS groups them into 8 themes (post-2023 framework).

In [ ]:
THEME_QUESTIONS = {
    'teaching':            [1, 2, 3, 4],
    'learning_opps':       [5, 6, 7, 8],
    'assessment_feedback': [9, 10, 11, 12, 13],
    'academic_support':    [14, 15, 16],
    'organisation':        [17, 18, 19],
    'learning_resources':  [20, 21, 22],
    'student_voice':       [23, 24, 25],
    'free_expression':     [26],
}

def question_to_theme(q):
    if pd.isna(q):
        return None
    for theme, qs in THEME_QUESTIONS.items():
        if int(q) in qs:
            return theme
    return None

## 3. Read one NSS CSV

OfS files have a 1–4 row title block before the real header. We scan the first 10 rows for one that contains both `'Question'` and `'Subject'` and use that as the header.

In [ ]:
def detect_header_row(path, max_scan=10):
    probe = pd.read_csv(path, header=None, nrows=max_scan, dtype=str, encoding_errors='replace')
    for i in range(len(probe)):
        cells = {str(c).strip() for c in probe.iloc[i].dropna().tolist()}
        if 'Question' in cells and 'Subject' in cells:
            return i
    return 0

def infer_year_and_cut(path):
    name = path.stem
    year_match = re.search(r'NSS(\d{2})', name)
    year = (2000 + int(year_match.group(1))) if year_match else None
    cut_match = re.search(r'_([rt])$', name)
    cut = cut_match.group(1) if cut_match else None
    return year, cut

def load_one_file(path):
    year, cut = infer_year_and_cut(path)
    header_row = detect_header_row(path)
    df = pd.read_csv(path, header=header_row, low_memory=False, encoding_errors='replace')
    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]

    df['question_num']   = df['Question'].astype(str).str.extract(r'Q(\d{1,2})', expand=False).astype('Int64')
    df['theme']          = df['question_num'].apply(question_to_theme)
    df['positivity_pct'] = pd.to_numeric(df.get('Positivity measure (%)'), errors='coerce')
    df['nss_year']       = year
    df['cut']            = cut
    return df

# Quick peek at one file.
sample = load_one_file(nss_files[0])
print(f'{nss_files[0].name}: {len(sample):,} rows')
print('Columns:', list(sample.columns)[:10])

## 4. Load every file and split into _r vs _t

In [ ]:
r_frames, t_frames = [], []
for path in nss_files:
    df = load_one_file(path)
    print(f'  {path.name}: {len(df):,} rows')
    if df['cut'].iloc[0] == 'r':
        r_frames.append(df)
    elif df['cut'].iloc[0] == 't':
        t_frames.append(df)

nss_r = pd.concat(r_frames, ignore_index=True)
nss_t = pd.concat(t_frames, ignore_index=True)
print(f'\nCombined registered: {len(nss_r):,} rows')
print(f'Combined taught-at:  {len(nss_t):,} rows')

## 5. Filter

Keep rows that are: **full-time**, **all undergraduates**, **CAH2 subject level**, with a real question number and a real positivity score.

We also strip whitespace from `Subject` so `'English studies '` and `'English studies'` collapse to one subject.

In [ ]:
def filter_nss(df):
    if df.empty:
        return df
    mode      = df['Mode of study'].astype(str).str.lower().str.strip()
    level     = df['Level of study'].astype(str).str.lower()
    subj_lvl  = df['Subject level'].astype(str).str.upper().str.strip()

    out = df[
        mode.isin(['full-time', 'fulltime', 'full time'])
        & level.str.contains('undergrad', na=False)
        & (subj_lvl == 'CAH2')
        & df['question_num'].notna()
        & df['positivity_pct'].notna()
    ].copy()

    # FIX from v1: strip stray trailing whitespace in Subject names.
    out['Subject'] = out['Subject'].astype(str).str.strip()
    return out

nss_r_f = filter_nss(nss_r)
nss_t_f = filter_nss(nss_t)
print(f'Registered after filter: {len(nss_r_f):,} rows')
print(f'Taught-at  after filter: {len(nss_t_f):,} rows')

print('\nCAH2 subjects present (de-duplicated):')
for s in sorted(nss_r_f['Subject'].dropna().unique()):
    print(f'  - {s!r}')

## 6. Average questions up to themes

Within each (year, subject, theme), take the response-weighted mean of the question-level positivity scores.

In [ ]:
def aggregate_to_theme(df):
    if df.empty:
        return df
    df = df.copy()
    df['n_responses'] = pd.to_numeric(df.get('Responses'), errors='coerce').fillna(1)
    df['weighted']    = df['positivity_pct'] * df['n_responses']

    out = (
        df.groupby(['nss_year', 'Subject code', 'Subject', 'theme'], as_index=False)
          .agg(weighted_sum=('weighted', 'sum'),
               weight_total=('n_responses', 'sum'),
               n_questions=('positivity_pct', 'count'))
    )
    out['positivity_pct'] = out['weighted_sum'] / out['weight_total']
    return out[['nss_year', 'Subject code', 'Subject', 'theme', 'positivity_pct', 'n_questions']]

nss_r_themes = aggregate_to_theme(nss_r_f)
nss_t_themes = aggregate_to_theme(nss_t_f)
print(f'Registered theme rows: {len(nss_r_themes):,}')
print(f'Taught-at  theme rows: {len(nss_t_themes):,}')

## 7. Pivot to wide form (one row per subject-year, eight theme columns)

In [ ]:
def to_wide(themes):
    t = themes.rename(columns={'Subject code': 'subject_code', 'Subject': 'subject'})
    wide = t.pivot_table(
        index=['nss_year', 'subject_code', 'subject'],
        columns='theme', values='positivity_pct', aggfunc='mean',
    ).reset_index()
    wide.columns.name = None
    theme_cols = [c for c in wide.columns if c not in ('nss_year', 'subject_code', 'subject')]
    return wide.rename(columns={c: f'nss_{c}' for c in theme_cols})

wide_r = to_wide(nss_r_themes)
wide_t = to_wide(nss_t_themes)
print(f'Registered wide: {len(wide_r):,} rows')
print(f'Taught-at  wide: {len(wide_t):,} rows')

## 8. Save

In [ ]:
long_r = nss_r_themes.rename(columns={'Subject code': 'subject_code', 'Subject': 'subject'})
long_t = nss_t_themes.rename(columns={'Subject code': 'subject_code', 'Subject': 'subject'})

long_r.to_csv(OUT_DIR / 'nss_exeter_long.csv',           index=False)
wide_r.to_csv(OUT_DIR / 'nss_exeter_wide.csv',           index=False)
long_t.to_csv(OUT_DIR / 'nss_exeter_taughtat_long.csv',  index=False)
wide_t.to_csv(OUT_DIR / 'nss_exeter_taughtat_wide.csv',  index=False)

print('Wrote:')
for n in ['nss_exeter_long.csv', 'nss_exeter_wide.csv',
          'nss_exeter_taughtat_long.csv', 'nss_exeter_taughtat_wide.csv']:
    p = OUT_DIR / n
    print(f'  {p}  ({p.stat().st_size/1024:.1f} KB)')

## 9. Department → NSS-subject mapping  *(string fixes from v1 + PSY fix)*

Each row of the mapping is `(dept_code, nss_subject)`. The NSS subject string MUST match the OfS string exactly, otherwise the merge in notebook 05 silently drops the dept.

**v1 problems I am fixing:**
- `GEO` → `'Geographical and environmental studies'`  ✗  (NSS uses `'Geography, earth and environmental studies'`)
- `EFP` (Education) had no NSS counterpart in the OfS data — removed.
- `LIB` (Liberal arts) had no NSS counterpart — removed.

**Late fix (after first v2 run):**
- **`PSY` (the main UG Psychology dept, 333 modules) was missing from the mapping**, so all the mainstream UG psychology data was silently dropped. v1 only mapped `PYC`, which turns out to be a small Stage-3 specialist programme. Both are now mapped to `'Psychology'`.

In [ ]:
MAPPING = [
    # ----- Business / Economics -----
    ('BEE',  'Economics'),
    ('BEM',  'Business and management'),
    ('BEA',  'Business and management'),
    ('BEF',  'Business and management'),
    ('BEP',  'Business and management'),
    # ----- STEM -----
    ('ECM',  'Computing'),
    ('ECM',  'Engineering'),
    ('ENG',  'Engineering'),
    ('MTH',  'Mathematical sciences'),
    ('PHY',  'Physics and astronomy'),
    ('GEO',  'Geography, earth and environmental studies'),  # FIX: was 'Geographical and environmental studies'
    ('BIO',  'Biosciences'),
    ('NSC',  'General, applied and forensic sciences'),
    # ----- Health -----
    ('PSY',  'Psychology'),  # FIX: PSY is the main UG Psychology dept (333 modules); was missing in v2-initial
    ('PYC',  'Psychology'),  # specialised Stage-3 child & adolescent mental health programme
    ('SHS',  'Sport and exercise sciences'),
    ('MED',  'Medicine and dentistry'),
    ('MDC',  'Medicine and dentistry'),
    ('NUR',  'Nursing and midwifery'),
    # ----- Humanities & Social Sciences -----
    ('EAS',  'English studies'),
    ('EAF',  'Media, journalism and communications'),
    ('HIH',  'History and archaeology'),
    ('ARC',  'History and archaeology'),
    ('POL',  'Politics'),
    ('SOC',  'Sociology, social policy and anthropology'),
    ('SPA',  'Sociology, social policy and anthropology'),
    ('ANT',  'Sociology, social policy and anthropology'),
    ('LAW',  'Law'),
    ('PHL',  'Philosophy and religious studies'),
    ('THE',  'Philosophy and religious studies'),
    ('CLA',  'Languages and area studies'),
    ('MLF',  'Languages and area studies'),
    ('MLG',  'Languages and area studies'),
    ('MLS',  'Languages and area studies'),
    ('MLI',  'Languages and area studies'),
    ('DRA',  'Performing arts'),
    # ----- Removed in v2: no matching NSS subject in the OfS data -----
    # ('EFP',  'Education and teaching'),
    # ('LIB',  'Liberal arts'),
]

mapping = pd.DataFrame(MAPPING, columns=['dept_code', 'nss_subject'])
mapping_path = OUT_DIR / 'dept_to_nss_subject.csv'
mapping.to_csv(mapping_path, index=False)
print(f'Wrote: {mapping_path}  ({len(mapping)} rows)')

# Sanity-check: every NSS subject in the mapping must exist in the data.
actual_subjects = set(wide_r['subject'].dropna())
mapped_subjects = set(mapping['nss_subject'])
missing = mapped_subjects - actual_subjects
print(f'\nMapping subjects NOT in NSS data: {sorted(missing) if missing else "(none)"}')
print(f'NSS subjects NOT used by any dept: {sorted(actual_subjects - mapped_subjects)}')

Next: **`05_merge_datasets.ipynb`** — join the dept-year module summary to the NSS wide table.